# 02 · Rewards lab

**Teach a Small AI to Think** · unicodesnwrites.tech

You will write the two reward functions that DeepSeek-R1-style training relies on, a **format** reward and a **correctness** reward, and test them on hand-written sample answers before any model gets near them.

**How to open it in Colab:** go to [colab.research.google.com](https://colab.research.google.com), choose **File → Upload notebook**, and pick this file. No GPU needed: the default CPU runtime is fine. You can also run it in Jupyter on your own computer.

**Expected runtime:** under a minute. Everything here uses only Python's standard library.

## 1. The shape we want

We will ask the model to answer like this:

```
<think> ...working... </think>
<answer> 18 </answer>
```

A **regular expression** (regex) is a tiny search pattern. The one below says: a `<think>` block, then an `<answer>` block, and nothing else around them.

In [ ]:
import re

FORMAT_PATTERN = re.compile(r"^\s*<think>.+?</think>\s*<answer>.+?</answer>\s*$", re.DOTALL)
ANSWER_PATTERN = re.compile(r"<answer>(.*?)</answer>", re.DOTALL)
NUMBER_PATTERN = re.compile(r"-?\d+(?:\.\d+)?")

## 2. Small helpers

- `completion_text` copes with both plain strings and chat messages, the two shapes TRL can pass in.
- `gold_from_gsm8k` keeps the number after `####`.
- `extract_answer` pulls out what is inside `<answer>`.
- `parse_number` keeps the last number, so `$1,200.` becomes `1200.0`.

In [ ]:
def completion_text(completion):
    if isinstance(completion, str):
        return completion
    return completion[0]["content"]

def gold_from_gsm8k(solution):
    return solution.split("####")[-1].strip().replace(",", "")

def extract_answer(text):
    match = ANSWER_PATTERN.search(text)
    return match.group(1).strip() if match else None

def parse_number(text):
    if text is None:
        return None
    numbers = NUMBER_PATTERN.findall(text.replace(",", ""))
    return float(numbers[-1]) if numbers else None

print(parse_number("$1,200."), parse_number("about 18 dollars"), parse_number("no idea"))

## 3. The reward functions

Each one takes a **list** of completions and returns a **list** of scores, one per completion. That is exactly what TRL's `GRPOTrainer` expects. The `**kwargs` catch any extra arguments TRL passes that we do not need.

In [ ]:
def format_reward(completions, **kwargs):
    return [1.0 if FORMAT_PATTERN.match(completion_text(c)) else 0.0 for c in completions]

def soft_format_reward(completions, **kwargs):
    tags = ["<think>", "</think>", "<answer>", "</answer>"]
    return [sum(0.25 for t in tags if completion_text(c).count(t) == 1) for c in completions]

def correctness_reward(completions, answer, **kwargs):
    scores = []
    for c, gold in zip(completions, answer):
        predicted = parse_number(extract_answer(completion_text(c)))
        target = parse_number(gold_from_gsm8k(gold))
        ok = predicted is not None and target is not None and abs(predicted - target) < 1e-6
        scores.append(1.0 if ok else 0.0)
    return scores

## 4. Grade some sample answers

These six answers are written by hand to poke at the edges. Before you run the cell, guess each score.

In [ ]:
gold = "She sells 16 - 3 - 4 = 9 eggs, earning 9 * 2 = 18 dollars.\n#### 18"
samples = {
    "tidy and right": "<think>16 - 3 - 4 = 9 eggs, 9 * 2 = 18</think><answer>18</answer>",
    "right, with $": "<think>9 eggs at $2 each</think>\n<answer>$18.</answer>",
    "tidy but wrong": "<think>16 - 3 = 13, 13 * 2 = 26</think><answer>26</answer>",
    "right, no tags": "The answer is 18.",
    "tags twice": "<think>hmm</think><answer>18</answer><answer>18</answer>",
    "chat message": [{"role": "assistant", "content": "<think>9 * 2</think><answer>18</answer>"}],
}

def report(samples, gold, fmt_fn=format_reward):
    completions = list(samples.values())
    answers = [gold] * len(completions)
    fmt = fmt_fn(completions)
    soft = soft_format_reward(completions)
    right = correctness_reward(completions, answers)
    print(f"{'sample':<16}{'format':>8}{'soft':>7}{'correct':>9}")
    for name, f, s, r in zip(samples, fmt, soft, right):
        print(f"{name:<16}{f:>8.2f}{s:>7.2f}{r:>9.2f}")

report(samples, gold)

## 5. Spot the loophole

Look at the `tags twice` row. The strict format reward gives it **1.0**, even though the answer block appears twice. The lazy `.+?` is allowed to stretch over the extra tags to reach the end of the text. A model under training pressure will find holes like this. Your job is to find them first.

## 6. Your turn: a stricter format reward

Write `strict_format_reward` so that:

- it returns 1.0 only when each of the four tags appears **exactly once**, in the right order, and the whole text matches the pattern;
- `tags twice` scores 0.0, and every other row keeps its old format score.

The tests in the next cell will print `All tests passed` when you are done.

In [ ]:
def strict_format_reward(completions, **kwargs):
    scores = []
    for c in completions:
        text = completion_text(c)
        # TODO: replace this line with your stricter check
        scores.append(1.0 if FORMAT_PATTERN.match(text) else 0.0)
    return scores

In [ ]:
def check(fn):
    got = dict(zip(samples, fn(list(samples.values()))))
    expected = {"tidy and right": 1.0, "right, with $": 1.0, "tidy but wrong": 1.0,
                "right, no tags": 0.0, "tags twice": 0.0, "chat message": 1.0}
    bad = {k: got[k] for k in expected if got[k] != expected[k]}
    print("All tests passed" if not bad else f"Not yet. Wrong scores: {bad}")

check(strict_format_reward)

### One possible solution

Try it yourself first. When you are ready, run the cell below to see one answer: it counts each tag before trusting the pattern.

In [ ]:
def strict_format_reward_solution(completions, **kwargs):
    tags = ["<think>", "</think>", "<answer>", "</answer>"]
    scores = []
    for c in completions:
        text = completion_text(c)
        once_each = all(text.count(t) == 1 for t in tags)
        scores.append(1.0 if once_each and FORMAT_PATTERN.match(text) else 0.0)
    return scores

check(strict_format_reward_solution)
report(samples, gold, fmt_fn=strict_format_reward_solution)

## 7. Try to break it

Add your own sneaky answers to `samples` and rerun section 4. Some ideas:

- `<think></think><answer>18</answer>`: empty thinking. Should that earn the format reward?
- `<think>...</think><answer>18 or 26</answer>`: two numbers. Which one does `parse_number` keep?
- `<answer>18</answer><think>done</think>`: right parts, wrong order.

Each surprise you find here is one less surprise during a training run.

In [ ]:
samples["two numbers"] = "<think>not sure</think><answer>18 or 26</answer>"
samples["wrong order"] = "<answer>18</answer><think>done</think>"
report(samples, gold, fmt_fn=strict_format_reward_solution)

## 8. How this plugs into TRL (preview)

In Module 5 you will hand these functions straight to the trainer. You do not need to run this here; it is just to show the shape:

```python
from trl import GRPOConfig, GRPOTrainer

trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=[strict_format_reward_solution, correctness_reward],
    args=GRPOConfig(output_dir="out"),
    train_dataset=dataset,  # needs a "prompt" column and an "answer" column
)
```

TRL adds the scores from all reward functions together (or a weighted sum if you set `reward_weights`).

Next lesson: **When rewards go wrong: reward hacking**.